# Practice 61 — pandas + NumPy

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns

---

## Level 1 — Departmental budget vs actual

`budget` tracks planned vs actual spend ($k) across three departments and two regions over three quarters.

1. Reshape to long format — one row per `(dept, region, quarter)` with `budget` and `actual` as separate columns.
2. Add `variance = actual - budget`. Use `np.percentile` to find the 10th and 90th percentile of variance across all rows. What range does most spending fall within?
3. Use `np.where` to label each row `'over'` or `'under'` (over if `actual > budget`). Which `region` goes over budget more often?

In [12]:
budget = pd.DataFrame({
    'dept':      ['Engineering', 'Marketing', 'Sales', 'Engineering', 'Marketing', 'Sales'],
    'region':    ['East', 'East', 'East', 'West', 'West', 'West'],
    'budget_Q1': [120, 80, 95, 100, 70, 85],
    'budget_Q2': [130, 85, 100, 110, 75, 90],
    'budget_Q3': [125, 90, 105, 115, 80, 95],
    'actual_Q1': [115, 92, 98,  108, 68, 87],
    'actual_Q2': [138, 82, 112, 105, 79, 95],
    'actual_Q3': [122, 95, 101, 120, 77, 100],
})

# Your code here

budget_long = pd.wide_to_long(
    budget, 
    stubnames=['budget','actual'],
    i = ['dept','region'],
    j = 'quarter',
    sep = '_',
    suffix= r'\w+'
)
budget_long['variance'] = budget_long['actual'] - budget_long['budget']
print(np.percentile(budget_long['variance'], q=[10,90]))
budget_long['act'] = np.where(budget_long['actual']> budget_long['budget'], 'over','under')
print(budget_long.groupby(level='region')['act'].apply(lambda x: (x=='over').mean()).idxmax(),'goes over more often')

[-4.3  9.2]
West goes over more often


---

## Level 2 — Penguins

Use `sns.load_dataset('penguins')`. Drop rows with any nulls first.

1. Crosstab of `island` vs `species`, `margins=True`. Which island is the most species-diverse (closest to an even split across the species it has)?
2. Normalize by row. Which island has the highest proportion of Gentoo penguins?
3. Add `bill_ratio = bill_length_mm / bill_depth_mm`. For each `species`, use `np.percentile` to report the 25th, 50th, and 75th percentile of `bill_ratio`. Which species has the widest IQR?

In [28]:
penguins = sns.load_dataset('penguins')

# Your code here
penguins = sns.load_dataset('penguins').dropna()
cp = pd.crosstab(penguins['island'], penguins['species'], margins=True)
print(pd.crosstab(penguins['island'], penguins['species'], normalize='index').std(axis = 1).idxmin(), 'is the most diverse')

pn = pd.crosstab(penguins['island'], penguins['species'], normalize='index')
print(pn['Gentoo'].idxmax(),'has the highest proportino of Gentoo')
penguins['bill_ratio'] = penguins['bill_length_mm']/penguins['bill_depth_mm']
pg = penguins.groupby('species', observed=True)['bill_ratio'].apply(lambda x: np.percentile(x, [25, 50, 75]))
iqr = pg.apply(lambda x: x[2] - x[0])
print(iqr.idxmax(),'has the widest IQR')

Dream is the most diverse
Biscoe has the highest proportino of Gentoo
Adelie has the widest IQR


---

## Level 3 — Exoplanets

Use `sns.load_dataset('planets')`. Columns include `method`, `number`, `orbital_period`, `mass`, `distance`, `year` (1989–2014). No sub-steps.

1. Bin `year` into three eras — you decide the cut points. Build a crosstab of `method` vs era, `margins=True`. Which detection method dominated the most recent era?
2. Drop rows where `mass` or `orbital_period` is null. Use `np.corrcoef` to check if heavier planets tend to have longer orbital periods.
3. Use `np.argsort` to identify the 5 most massive planets. Which detection `method` found them?

In [48]:
planets = sns.load_dataset('planets')

# Your code here
planets['era'] = pd.cut(planets['year'], bins=3, labels=['old','early','current'])
p = pd.crosstab(planets['method'],planets['era'], margins=True)
print(p.loc[p.index != 'All', 'current'].idxmax(), 'dominate the most recent era')

planets2 = planets.dropna(subset=['mass','orbital_period'])
np.corrcoef(planets2['mass'],planets2['orbital_period'])[0,1]
print('weakly correlated')
planets3 = planets.dropna(subset=['mass'])
sorted_p = planets3.iloc[np.argsort(-planets3['mass'])]
heavest_five = sorted_p.iloc[:5]
print(heavest_five)
print(heavest_five['method'].unique(),'identified them')


Transit dominate the most recent era
weakly correlated
              method  number  orbital_period   mass  distance  year      era
321  Radial Velocity       1         2371.00  25.00     37.05  2008  current
85   Radial Velocity       2          379.63  21.42       NaN  2009  current
63   Radial Velocity       1          305.50  20.60     92.51  2013  current
913  Radial Velocity       1          677.80  19.80       NaN  2007  current
3    Radial Velocity       1          326.03  19.40    110.62  2007  current
['Radial Velocity'] identified them
